# 07 — Analisis Temporal Sentimen + Toksisitas (2016-2026)

Time-series per-bulan + per-tahun, plot dengan event overlay (patch major, era pandemi, TI, era DPC), Mann-Kendall trend test, Chow structural break test.

**Pra-syarat**: notebook 05 selesai, `data/inference/` terisi.

**Output**: `reports/temporal_{monthly,yearly,tests,event_comparison}.csv` + `reports/plots/temporal_*.{png,svg}`.

In [1]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('07_temporal_analysis', config)
run_log = RunLog(notebook='07_temporal_analysis', config_path='configs/experiment.yaml')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED_ROOT = Path(config['data']['processed_root'])
INF_ROOT = Path(config['data']['inference_root'])
REPORTS = Path('reports')
PLOTS = REPORTS / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

SENT_LABELS = config['labels']['sentiment_classes']
TOX_LABELS = config['labels']['toxicity_labels']

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 07_temporal_analysis
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: da158c0
Started at: 2026-06-09T10:41:19+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [2]:
# Sel 2: Pilih best model — default RoBERTa untuk sentimen + Detoxify untuk toksisitas.
# Pilihan ini didokumentasikan di reports/comparison_summary.md (notebook 06).
BEST_SENT_MODEL = 'roberta'
BEST_TOX_MODEL = 'detoxify'
print(f'Best sentiment model: {BEST_SENT_MODEL}')
print(f'Best toxicity model:  {BEST_TOX_MODEL}')

from src.analysis.temporal import merge_inference_with_processed
df = merge_inference_with_processed(
    processed_root=PROCESSED_ROOT,
    sentiment_inference_root=INF_ROOT / f'{BEST_SENT_MODEL}_sentiment',
    toxicity_inference_root=INF_ROOT / f'{BEST_TOX_MODEL}_toxicity',
    sentiment_labels=SENT_LABELS,
    toxicity_labels=TOX_LABELS,
)
print(f'Total rows merged: {len(df):,}')

Best sentiment model: roberta
Best toxicity model:  detoxify
Total rows merged: 1,380,867


In [3]:
# Sel 3: Agregat bulanan + tahunan
from src.analysis.temporal import aggregate_monthly, aggregate_yearly

monthly = aggregate_monthly(df, sentiment_labels=SENT_LABELS, toxicity_labels=TOX_LABELS)
yearly = aggregate_yearly(df, sentiment_labels=SENT_LABELS, toxicity_labels=TOX_LABELS)
monthly.to_csv(REPORTS / 'temporal_monthly.csv', index=False)
yearly.to_csv(REPORTS / 'temporal_yearly.csv', index=False)
print(f'monthly: {len(monthly)} bins  |  yearly: {len(yearly)} bins')
run_log.add_output(REPORTS / 'temporal_monthly.csv')
run_log.add_output(REPORTS / 'temporal_yearly.csv')
monthly.head()

monthly: 124 bins  |  yearly: 11 bins


,year_month,n_messages,n_matches,pct_negative,pct_neutral,pct_positive,mean_sentiment_score,pct_toxic_any,mean_toxicity_score,pct_toxic,pct_severe_toxic,pct_obscene,pct_threat,pct_insult,pct_identity_hate
0,2016-01,5576,549,0.019010,0.937948,0.043042,0.034586,0.012016,0.052023,0.012016,0.000000,0.004484,0.000359,0.001973,0.000000
1,2016-02,6134,611,0.020867,0.944408,0.034724,0.028580,0.008966,0.048251,0.008966,0.000000,0.002608,0.000163,0.000163,0.000000
2,2016-03,6545,592,0.028877,0.924370,0.046753,0.028790,0.009473,0.046297,0.009473,0.000306,0.003514,0.000306,0.001528,0.000611
3,2016-04,5132,502,0.022603,0.927903,0.049493,0.038442,0.006625,0.045548,0.006625,0.000000,0.001949,0.000000,0.000585,0.000195
4,2016-05,12137,1142,0.030485,0.921315,0.048200,0.031686,0.012606,0.046788,0.012606,0.000165,0.004696,0.000082,0.002060,0.000247


In [4]:
# Sel 4: Plot time-series dengan event overlay
from src.analysis.event_overlay import major_patches, the_internationals, pandemic_band, dpc_era

monthly['_dt'] = pd.to_datetime(monthly['year_month'])
patches = major_patches()
tis = the_internationals()
p_start, p_end = pandemic_band()
dpc_start, dpc_end = dpc_era()

def add_overlay(ax):
    ax.axvspan(dpc_start, dpc_end, alpha=0.06, color='blue', label='DPC era')
    ax.axvspan(p_start, p_end, alpha=0.10, color='orange', label='Pandemic (online)')
    for ev in patches:
        ax.axvline(ev.date, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
        ax.text(ev.date, ax.get_ylim()[1], ev.name, rotation=90, fontsize=7, va='top', alpha=0.6)
    for ev in tis:
        ax.axvline(ev.date, color='red', linestyle=':', alpha=0.5, linewidth=0.8)

for col, title, fname in [
    ('mean_sentiment_score', 'Mean Sentiment Score (P(pos) - P(neg))', 'temporal_sentiment_monthly'),
    ('pct_negative', '% Negative Messages', 'temporal_negative_pct_monthly'),
    ('pct_toxic_any', '% Toxic Messages (any label)', 'temporal_toxicity_monthly'),
    ('mean_toxicity_score', 'Mean Toxicity Score (max prob)', 'temporal_toxicity_score_monthly'),
]:
    if col not in monthly.columns:
        continue
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(monthly['_dt'], monthly[col], linewidth=1.5)
    add_overlay(ax)
    ax.set_title(title); ax.set_xlabel('Month'); ax.set_ylabel(col)
    ax.legend(loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOTS / f'{fname}.png', dpi=120)
    plt.savefig(PLOTS / f'{fname}.svg')
    plt.close(fig)
    run_log.add_output(PLOTS / f'{fname}.png')

# Yearly bar chart
fig, ax = plt.subplots(figsize=(10, 4))
if 'mean_sentiment_score' in yearly.columns:
    ax.bar(yearly['year'], yearly['mean_sentiment_score'], alpha=0.8)
    ax.set_title('Mean Sentiment Score per Year'); ax.set_xlabel('Year')
plt.tight_layout()
plt.savefig(PLOTS / 'temporal_yearly_overview.png', dpi=120)
plt.savefig(PLOTS / 'temporal_yearly_overview.svg')
plt.close(fig)

In [5]:
# Sel 5: Mann-Kendall + Chow tests
from src.analysis.trend_tests import run_trend_suite, run_chow_suite

series_for_test = {
    name: monthly[name].values
    for name in ['mean_sentiment_score', 'pct_negative', 'pct_toxic_any', 'mean_toxicity_score']
    if name in monthly.columns
}
mk_df = run_trend_suite(series_for_test)

# Chow test breakpoints: indeks bulan kandidat (start of pandemic, post-DPC, patch 7.33)
def _idx_for_date(date_str: str) -> int:
    target = pd.Timestamp(date_str).to_period('M').strftime('%Y-%m')
    if target in monthly['year_month'].values:
        return int(monthly.index[monthly['year_month'] == target][0])
    # Fallback: closest available index
    return int(np.argmin(np.abs(monthly['_dt'].values - pd.Timestamp(date_str).to_datetime64())))

breakpoints = {
    'pandemic_start_2020-03': _idx_for_date('2020-03-01'),
    'post_dpc_2023-01': _idx_for_date('2023-01-01'),
    'patch_7.33_2023-04': _idx_for_date('2023-04-20'),
}
chow_df = run_chow_suite(series_for_test, breakpoints)

tests_df = pd.concat([mk_df, chow_df], ignore_index=True)
tests_df.to_csv(REPORTS / 'temporal_tests.csv', index=False)
print(tests_df.to_string(index=False))
run_log.add_output(REPORTS / 'temporal_tests.csv')

              series    test_name  test_statistic   z_score      p_value   interpretation   n       breakpoint_label  breakpoint_index  f_statistic
mean_sentiment_score Mann-Kendall       -0.578809 -9.531177 0.000000e+00       decreasing 124                    NaN               NaN          NaN
        pct_negative Mann-Kendall       -0.221610 -3.647894 2.643984e-04       decreasing 124                    NaN               NaN          NaN
       pct_toxic_any Mann-Kendall       -0.494624 -8.144588 4.440892e-16       decreasing 124                    NaN               NaN          NaN
 mean_toxicity_score Mann-Kendall        0.324154  5.336854 9.457302e-08       increasing 124                    NaN               NaN          NaN
mean_sentiment_score         Chow             NaN       NaN 2.330849e-02 structural_break 124 pandemic_start_2020-03              50.0     3.879182
mean_sentiment_score         Chow             NaN       NaN 7.074505e-01         no_break 124       post_dpc_202

In [6]:
# Sel 6: Event-comparison ringkasan (Mann-Whitney pairwise)
from scipy.stats import mannwhitneyu

df['_sdt'] = pd.to_datetime(df['start_date_time'], errors='coerce')
rows = []
comparisons = [
    ('pre-pandemic', '2016-01-01', '2020-02-29', 'pandemic-online', '2020-03-01', '2021-12-31'),
    ('pandemic-online', '2020-03-01', '2021-12-31', 'post-pandemic-LAN', '2022-01-01', '2026-12-31'),
    ('dpc-era', '2017-01-01', '2022-12-31', 'post-dpc', '2023-01-01', '2026-12-31'),
]
for nameA, sA, eA, nameB, sB, eB in comparisons:
    a = df[(df['_sdt'] >= sA) & (df['_sdt'] <= eA)]
    b = df[(df['_sdt'] >= sB) & (df['_sdt'] <= eB)]
    if 'max_toxicity_prob' in df.columns:
        ya = a['max_toxicity_prob'].dropna()
        yb = b['max_toxicity_prob'].dropna()
        if len(ya) > 30 and len(yb) > 30:
            stat, p = mannwhitneyu(ya, yb, alternative='two-sided')
            rows.append({
                'comparison': f'{nameA} vs {nameB}',
                'metric': 'max_toxicity_prob',
                'mean_A': float(ya.mean()), 'mean_B': float(yb.mean()),
                'n_A': len(ya), 'n_B': len(yb),
                'mannwhitney_u': float(stat), 'p_value': float(p),
            })
    if 'sentiment_score' in df.columns:
        ya = a['sentiment_score'].dropna()
        yb = b['sentiment_score'].dropna()
        if len(ya) > 30 and len(yb) > 30:
            stat, p = mannwhitneyu(ya, yb, alternative='two-sided')
            rows.append({
                'comparison': f'{nameA} vs {nameB}',
                'metric': 'sentiment_score',
                'mean_A': float(ya.mean()), 'mean_B': float(yb.mean()),
                'n_A': len(ya), 'n_B': len(yb),
                'mannwhitney_u': float(stat), 'p_value': float(p),
            })

ec_df = pd.DataFrame(rows)
ec_df.to_csv(REPORTS / 'event_comparison.csv', index=False)
print(ec_df.to_string(index=False) if not ec_df.empty else '(empty)')
run_log.add_output(REPORTS / 'event_comparison.csv')
run_log.save('reports/run_log.csv')

                          comparison            metric   mean_A   mean_B    n_A    n_B  mannwhitney_u       p_value
     pre-pandemic vs pandemic-online max_toxicity_prob 0.052540 0.050357 448580 248818   5.342505e+10 7.769958e-193
     pre-pandemic vs pandemic-online   sentiment_score 0.027736 0.025069 448580 248818   5.409107e+10 6.686006e-101
pandemic-online vs post-pandemic-LAN max_toxicity_prob 0.050357 0.052500 248818 682729   7.768714e+10  0.000000e+00
pandemic-online vs post-pandemic-LAN   sentiment_score 0.025069 0.017026 248818 682729   7.980793e+10  0.000000e+00
                 dpc-era vs post-dpc max_toxicity_prob 0.051851 0.053192 742315 545714   1.800257e+11  0.000000e+00
                 dpc-era vs post-dpc   sentiment_score 0.024965 0.016164 742315 545714   1.861413e+11  0.000000e+00
[run_log] 07_temporal_analysis → 14.96s, 8 outputs, 0 warnings → reports\run_log.csv
